# Matplotlib: `subplots`, wiele serii na jednej osi, `twinx`

**Problem:** trzy różne potrzeby często się mylą, choć to zupełnie różne mechanizmy: (1) **wiele osobnych wykresów** obok siebie w siatce, (2) **wiele serii danych na TEJ SAMEJ osi** (te same jednostki, np. dwie linie sprzedaży), (3) **dwie różne skale na jednej osi** (`twinx`, np. sprzedaż w PLN i liczba jednostek na tym samym wykresie).

**Porównanie:**
- `plt.subplots(nrows, ncols)` — siatka NIEZALEŻNYCH osi (`Axes`), każda z własną skalą.
- `ax.plot()` wywołane kilka razy na tym samym `ax` — kilka serii na wspólnej skali.
- `ax.twinx()`/`ax.twiny()` — druga, nakładająca się oś z własną skalą, współdzieląca tę samą przestrzeń wykresu.

**Kiedy stosować:** osobne subploty, gdy dane mają różny charakter/zakres i porównanie "jeden na jednym" byłoby czytelniejsze; kilka serii na jednej osi, gdy jednostki są wspólne (bezpośrednie porównanie wartości ma sens); `twinx`, gdy koniecznie chcesz pokazać dwie różne miary NA TYM SAMYM wykresie (np. korelację w czasie), mimo różnych jednostek — ale ostrożnie, bo łatwo o mylącą wizualizację (patrz Sekcja 4).

**Uwaga o stylu:** bez narzuconych customowych motywów — domyślny wygląd matplotlib, żeby styling było łatwo dostosować samodzielnie.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(9)
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
sales = rng.integers(800, 1500, 6)
units = rng.integers(20, 60, 6)
costs = rng.integers(400, 900, 6)

print(sales, units, costs)

## Sekcja 1 — Podstawy: `plt.subplots(nrows, ncols)`

Zwraca **dwie rzeczy naraz**: `fig` (cała figura, do zapisu/globalnych ustawień) i `axes` (pojedyncza oś albo tablica osi — patrz Pułapka 1 w sprawie kształtu tej tablicy).

**Jak czytać `nrows`/`ncols`:** `nrows=2, ncols=3` to siatka 2 wiersze × 3 kolumny = 6 niezależnych wykresów, indeksowanych jak macierz: `axes[0, 0]` to lewy-górny, `axes[1, 2]` to prawy-dolny. Kolejność wypełniania to kwestia Twojego kodu, nie czegoś narzuconego przez matplotlib.

**`figsize` vs `dpi`:** `figsize=(szerokość, wysokość)` w CALACH określa rozmiar figury na ekranie/w edytorze; `dpi` (przy `savefig`, Sekcja 8) mnoży to przez gęstość pikseli przy zapisie. Figura `figsize=(8,4)` zapisana z `dpi=150` da plik o wymiarach `1200×600` pikseli — te dwa parametry razem decydują o ostrości i rozmiarze pliku.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8, 3))

axes[0].plot(months, sales)
axes[0].set_title("Sprzedaż")

axes[1].bar(months, units)
axes[1].set_title("Jednostki")

### Siatka 2×3 — jak wygląda indeksowanie przy większej liczbie wymiarów

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(10, 5))

data_per_panel = [sales, units, costs, sales[::-1], units[::-1], costs[::-1]]
positions = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]

for (row, col), values in zip(positions, data_per_panel):
    axes[row, col].plot(months, values)
    axes[row, col].set_title(f"axes[{row},{col}]")

## Sekcja 2 — Kilka serii na TEJ SAMEJ osi

Każde kolejne wywołanie `ax.plot()`/`ax.bar()` na tym samym `ax` dokłada kolejną serię — nie tworzy nowego wykresu. **Trik:** matplotlib automatycznie przydziela kolejny kolor z domyślnego cyklu przy każdym wywołaniu — nie trzeba ręcznie ustawiać `color=`, chyba że zależy Ci na konkretnych kolorach.

In [ ]:
fig, ax = plt.subplots()
ax.plot(months, sales, label="Sprzedaż")
ax.plot(months, costs, label="Koszty")
ax.legend()

### Mieszanie typów wykresu na jednej osi: słupki + linia

Nic nie stoi na przeszkodzie, żeby na tym samym `ax` połączyć `ax.bar()` (np. wartość miesięczna) z `ax.plot()` (np. średnia krocząca albo cel) — o ile jednostki są wspólne. To częsty układ w raportach: słupki pokazują "surowe" dane, linia dokłada kontekst (trend/benchmark).

In [ ]:
target_line = [1100] * len(months)  # stały cel sprzedażowy do porównania

fig, ax = plt.subplots()
ax.bar(months, sales, label="Sprzedaż miesięczna", color="tab:blue", alpha=0.7)
ax.plot(months, target_line, label="Cel", color="tab:red", linestyle="--", linewidth=2)
ax.legend()

### `ax.fill_between()`: zaznaczenie obszaru, nie tylko linii

Przydatne do pokazania przedziału niepewności/odchylenia wokół głównej linii — wypełniony obszar między dwiema seriami (np. dolna/górna granica prognozy) czyta się szybciej niż dwie osobne linie.

In [ ]:
lower_bound = [s * 0.9 for s in sales]
upper_bound = [s * 1.1 for s in sales]

fig, ax = plt.subplots()
ax.plot(months, sales, color="tab:blue", label="Sprzedaż")
ax.fill_between(months, lower_bound, upper_bound, alpha=0.2, color="tab:blue", label="Przedział ±10%")
ax.legend()

## Sekcja 3 — `sharex` / `sharey`

Wspólna skala osi między subplotami — przewijanie/zoom na jednym przesuwa pozostałe (w trybie interaktywnym), a wizualnie ułatwia porównanie zakresów.

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(6, 5))
axes[0].plot(months, sales)
axes[0].set_title("Sprzedaż")
axes[1].bar(months, units)
axes[1].set_title("Jednostki")

## Sekcja 4 — Dwie skale na jednej osi: `twinx()`

`ax.twinx()` tworzy drugą oś Y nałożoną na tę samą przestrzeń wykresu, współdzielącą oś X. **Trik:** koloruj etykiety osi (`set_ylabel(..., color=...)`) tym samym kolorem co odpowiadająca linia — bez tego trudno się zorientować, która skala należy do której serii.

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(months, sales, color="tab:blue")
ax1.set_ylabel("Sprzedaż (PLN)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(months, units, color="tab:orange")
ax2.set_ylabel("Jednostki", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

**Pułapka wewnątrz triku:** `ax1.legend()` widzi TYLKO serie narysowane na `ax1` — druga oś ma osobny, niepowiązany system legendy. Trzeba ręcznie zebrać uchwyty (`handles`) z obu osi i połączyć je w jedną legendę.

In [ ]:
fig, ax1 = plt.subplots()

line1, = ax1.plot(months, sales, color="tab:blue", label="Sprzedaż")
ax1.set_ylabel("Sprzedaż (PLN)", color="tab:blue")

ax2 = ax1.twinx()
line2, = ax2.plot(months, units, color="tab:orange", label="Jednostki")
ax2.set_ylabel("Jednostki", color="tab:orange")

# Ręczne połączenie uchwytów z obu osi w jedną, wspólną legendę
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper left")

## Sekcja 5 — Nierówne układy: `gridspec_kw` i `subplot_mosaic`

`gridspec_kw={"width_ratios": [...]}` w `plt.subplots()` kontroluje względne proporcje wierszy/kolumn bez osobnego importu `GridSpec`. `subplot_mosaic()` idzie o krok dalej — układ podajesz jako "mapę" tekstową, a osie dostajesz jako słownik po nazwach zamiast pozycyjnie indeksowanej tablicy.

In [ ]:
fig, axes = plt.subplots(1, 2, gridspec_kw={"width_ratios": [3, 1]}, figsize=(8, 3))
axes[0].plot(months, sales)
axes[1].bar(["Suma"], [sum(sales)])

In [ ]:
fig, axd = plt.subplot_mosaic([
    ["main", "main"],
    ["left", "right"],
], figsize=(7, 5))

axd["main"].plot(months, sales)
axd["main"].set_title("Sprzedaż w czasie")
axd["left"].bar(months, units)
axd["left"].set_title("Jednostki")
axd["right"].bar(months, costs)
axd["right"].set_title("Koszty")

### Puste komórki: `"."`

Kropka w układzie mozaiki oznacza "tu NIE twórz osi" — przydatne do celowych pustych miejsc (np. miejsce na logo, albo asymetryczny układ bez dociągania wszystkiego do pełnej siatki).

In [ ]:
fig, axd = plt.subplot_mosaic([
    ["main", "main", "."],
    ["left", "mid", "right"],
], figsize=(9, 4))

print(f"Utworzone osie: {list(axd.keys())}")  # brak 'osi' dla '.'

## Sekcja 6 — Wspólna legenda dla całej figury

`fig.legend()` zamiast `ax.legend()` na każdej osi osobno — jedna legenda opisująca serie z wielu subplotów naraz, umieszczona względem całej figury (np. na dole, pod wszystkimi wykresami).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
line1, = axes[0].plot(months, sales, label="Sprzedaż")
line2, = axes[1].plot(months, costs, label="Koszty", color="tab:red")

fig.legend(handles=[line1, line2], loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.05))

## Sekcja 7 — Layout: `constrained_layout` (i kiedy nie `tight_layout`)

Długie tytuły/etykiety w siatce subplotów łatwo się na siebie nachodzą. `constrained_layout=True` (przekazane przy tworzeniu figury) rozwiązuje to na bieżąco, w miarę dodawania elementów — nowszy i zwykle bardziej niezawodny mechanizm niż wywoływane na końcu `fig.tight_layout()`.

In [ ]:
fig, axes = plt.subplots(2, 2, constrained_layout=True, figsize=(7, 5))
for ax in axes.flat:
    ax.plot(months, sales)
    ax.set_title("Bardzo długi tytuł, który mógłby się nachodzić na sąsiedni wykres")

## Sekcja 8 — Zapis do pliku

`fig.savefig()`, nie `plt.savefig()` — ta druga forma działa tylko na "aktualnie aktywnej" figurze, co bywa niejednoznaczne, gdy w pamięci jest ich kilka naraz (np. w pętli generującej wiele wykresów).

In [ ]:
fig, ax = plt.subplots()
ax.plot(months, sales)
fig.savefig("/tmp/przyklad.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Zapisano")

## Sekcja 9 — Alternatywy z innych bibliotek

- **`pandas.DataFrame.plot(subplots=True)`** — najszybsza droga do siatki wykresów wprost z `DataFrame`, jedna kolumna = jeden subplot, bez ręcznego `plt.subplots()`. Mniej kontroli niż czysty matplotlib, ale świetne do szybkiego podglądu.
- **seaborn `FacetGrid`** — gdy podział na subploty wynika z wartości KOLUMNY KATEGORYCZNEJ (nie z góry ustalonej liczby wykresów) — osobna, głębsza notatka w tym repo.
- **plotly (`make_subplots`)** — gdy potrzebna interaktywność (zoom, hover, tooltips) w środowisku, które to obsługuje (np. eksport do HTML/dashboardu).

In [ ]:
df = pd.DataFrame({"month": months, "sales": sales, "units": units}).set_index("month")
df.plot(subplots=True, figsize=(6, 4))

## Sekcja 10 — Pułapki

### Pułapka 1 — kształt `axes` zależy od `nrows`/`ncols`, a nie jest stały

`plt.subplots()` zwraca **różny typ** w zależności od układu: 2D tablicę `numpy` gdy oba wymiary > 1, 1D tablicę gdy jeden z nich = 1, a pojedynczy obiekt `Axes` (NIE tablicę) gdy oba = 1. Kod pisany pod jeden przypadek (`axes[i, j]`) wybuchnie na innym. Zabezpieczenie: `squeeze=False` wymusza zawsze 2D, niezależnie od układu.

In [ ]:
fig1, axes1 = plt.subplots(2, 2)
print(f"2x2 -> {type(axes1).__name__}, shape={axes1.shape}")
plt.close(fig1)

fig2, axes2 = plt.subplots(1, 3)
print(f"1x3 -> {type(axes2).__name__}, shape={axes2.shape}")
plt.close(fig2)

fig3, axes3 = plt.subplots(1, 1)
print(f"1x1 -> {type(axes3).__name__} (BRAK .shape - to nie tablica!)")
plt.close(fig3)

fig4, axes4 = plt.subplots(1, 1, squeeze=False)
print(f"1x1 z squeeze=False -> {type(axes4).__name__}, shape={axes4.shape} (zawsze 2D)")
plt.close(fig4)

### Pułapka 2 — `plt.subplot()` (liczba pojedyncza) to inne, starsze API niż `plt.subplots()`

Nazwa różni się jedną literą, a mechanizm jest inny: `plt.subplot(nrows, ncols, index)` tworzy JEDNĄ oś naraz, z pozycją liczoną **od 1** (nie od 0), i działa na niejawnej "aktualnej figurze" zamiast zwracać `fig` wprost. Łatwo pomylić z nowszym, zalecanym `plt.subplots()` (liczba mnoga), które zwraca od razu `fig` + całą tablicę osi.

In [ ]:
fig = plt.figure()
ax1 = plt.subplot(2, 2, 1)  # pozycja 1 z 4, liczona OD JEDEN
ax2 = plt.subplot(2, 2, 2)
ax1.plot(months, sales)
print(f"plt.subplot() (l.pojedyncza) zwraca pojedynczy obiekt: {type(ax1).__name__}")
plt.close(fig)

### Pułapka 3 — `plt.plot()` bez `ax.` działa na niejawnie "bieżącej" osi

Wywołania `plt.` (bez wskazania konkretnego `ax`) zawsze trafiają na tzw. bieżącą oś (`plt.gca()`) — czyli tę, która była OSTATNIO "dotknięta", niekoniecznie tę, którą właśnie widzisz w kodzie wyżej. Przy pracy z wieloma subplotami mieszanie stylu `plt.plot(...)` i `ax.plot(...)` w tym samym bloku kodu to prosta droga do wykresu narysowanego "nie tam, gdzie myślałeś". Zasada: przy więcej niż jednym subplocie ZAWSZE odwołuj się jawnie przez zmienną `ax`, nigdy przez gołe `plt.`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].plot(months, sales)  # jawnie: rysuj na LEWEJ osi

plt.plot(months, costs)  # BEZ wskazania osi - trafia na 'bieżącą', nie na axes[0]!

print(f"'Bieżąca' oś (plt.gca()) to axes[1], NIE axes[0]: {plt.gca() is axes[1]}")
plt.close(fig)

## Sekcja 11 — Pełny scenariusz: budowa dashboardu krok po kroku

Zadanie: miesięczny raport sprzedaży za I połowę roku, jeden obraz do wysłania mailem — trend w czasie, rozbicie na region i kategorię, plus panel z kluczowymi liczbami. To połączenie technik z całej notatki w jednym, realistycznym przypadku.

### Krok 1 — Zaplanuj układ na kartce, ZANIM napiszesz kod

Cztery panele o różnej roli, nie cztery równe kwadraty:
```
┌─────────────────────────────┐
│         trend (pełna         │   <- wiersz 1: trend w czasie, cała szerokość
│         szerokość)           │
├───────────┬────────┬─────────┤
│  region   │category│   kpi   │   <- wiersz 2: trzy węższe panele
└───────────┴────────┴─────────┘
```
Górny wiersz jest WYŻSZY niż dolny (trend zasługuje na więcej miejsca niż zestawienia) — to sygnał, że potrzebne `height_ratios`, nie równa siatka 2×3. `subplot_mosaic` z nazwanym układem pasuje tu lepiej niż `plt.subplots()` — łatwiej czytać kod, gdy panele mają nazwy, nie indeksy `[i, j]`.

### Krok 2 — Przygotuj dane i policz wskaźniki ZANIM zaczniesz rysować

Rysowanie i liczenie w jednej linii utrudnia debugowanie — łatwiej sprawdzić `avg_monthly` osobno, niż szukać błędu w gąszczu wywołań `ax.text()`.

In [ ]:
sales_trend = [92000, 95000, 101000, 98000, 108000, 115000]
region_names = ["North", "South", "East", "West"]
region_sales = [285000, 240000, 198000, 210000]
category_names = ["Elektronika", "Akcesoria", "Serwis"]
category_sales = [420000, 310000, 190000]

total_sales = sum(sales_trend)
avg_monthly = np.mean(sales_trend)
growth_pct = (sales_trend[-1] - sales_trend[0]) / sales_trend[0] * 100
best_region = region_names[np.argmax(region_sales)]

print(f"Suma: {total_sales:,.0f} | Średnia: {avg_monthly:,.0f} | Wzrost: {growth_pct:.1f}% | Lider: {best_region}")

### Krok 3 — Zbuduj szkielet figury: `subplot_mosaic` + `height_ratios` + `constrained_layout`

Wszystkie trzy ustawienia globalne (rozmiar, proporcje wierszy, mechanizm odstępów) ustala się RAZEM, przy tworzeniu figury — nie doklejamy ich później.

In [ ]:
fig, axd = plt.subplot_mosaic(
    [
        ["trend", "trend", "trend"],
        ["region", "category", "kpi"],
    ],
    figsize=(11, 6),
    height_ratios=[2, 1.2],       # górny panel wyraźnie wyższy niż dolny rząd
    constrained_layout=True,       # automatyczne odstępy, bez ręcznego dostrajania
)
print(f"Utworzone panele: {list(axd.keys())}")

### Krok 4 — Wypełnij panele PO KOLEI, jeden na raz

Każdy panel to osobny, samodzielny blok: dobierz typ wykresu do treści (trend → linia, porównanie kategorii → słupki), nie kopiuj tego samego typu wszędzie z przyzwyczajenia. Panel `kpi` w ogóle nie jest wykresem — `ax.axis("off")` wyłącza osie/ramkę, `ax.text()` renderuje czysty tekst w środku.

In [ ]:
ax = axd["trend"]
ax.plot(months, sales_trend, marker="o", linewidth=2)
ax.set_title("Trend sprzedaży 2026 (PLN)")
ax.set_ylabel("Sprzedaż (PLN)")
ax.grid(True, alpha=0.3)

In [ ]:
ax = axd["region"]
ax.bar(region_names, region_sales, color="tab:blue")
ax.set_title("Sprzedaż wg regionu")
ax.tick_params(axis="x", rotation=30)  # dłuższe nazwy - obrót, żeby się nie nachodziły

In [ ]:
ax = axd["category"]
ax.barh(category_names, category_sales, color="tab:orange")  # poziome - nazwy kategorii czytelniejsze
ax.set_title("Sprzedaż wg kategorii")

In [ ]:
ax = axd["kpi"]
ax.axis("off")  # brak wykresu - tylko tekst, więc wyłączamy osie/ramkę

kpi_text = (
    f"Suma H1: {total_sales:,.0f} PLN\n"
    f"Śr. miesięczna: {avg_monthly:,.0f} PLN\n"
    f"Wzrost Jan→Jun: {growth_pct:.1f}%\n"
    f"Najlepszy region: {best_region}"
)
ax.text(0.05, 0.5, kpi_text, fontsize=11, va="center", family="monospace")
ax.set_title("Kluczowe wskaźniki")

### Krok 5 — Tytuł całości i zapis

`fig.suptitle()` (nie `ax.set_title()` na pojedynczym panelu) dla tytułu obejmującego całą figurę. Zapis na samym końcu, gdy wszystkie panele są już gotowe.

In [ ]:
fig.suptitle("Raport sprzedaży — I połowa 2026", fontsize=14, fontweight="bold")
fig.savefig("/tmp/dashboard_sprzedazowy.png", dpi=150, bbox_inches="tight")
print("Zapisano dashboard_sprzedazowy.png")

### Podsumowanie scenariusza — kolejność decyzji, nie tylko kodu

1. **Układ na kartce** przed kodem — które panele są ważniejsze, czy potrzebne różne proporcje wierszy/kolumn.
2. **Dane i wskaźniki policzone osobno** od rysowania — łatwiej debugować.
3. **Szkielet figury ustawiony za jednym razem** (`figsize`, `height_ratios`, `constrained_layout`) — nie doklejany później.
4. **Panele wypełniane po kolei**, typ wykresu dobrany do treści, nie "zawsze to samo".
5. **Tytuł całości i zapis na końcu**, gdy zawartość jest już gotowa — zmiana układu po fakcie zwykle wymaga przebudowy `figsize`/`height_ratios` od nowa.

## Podsumowanie

| Zadanie | Rozwiązanie |
|---|---|
| Siatka niezależnych wykresów | `fig, axes = plt.subplots(nrows, ncols)` |
| Zawsze 2D tablica osi, niezależnie od układu | `squeeze=False` |
| Kilka serii na jednej osi | wielokrotne `ax.plot()`/`ax.bar()` na tym samym `ax` |
| Wspólna skala między subplotami | `sharex=True` / `sharey=True` |
| Druga skala Y na tej samej osi | `ax.twinx()` (+ ręczne połączenie legend) |
| Nierówne proporcje wierszy/kolumn | `gridspec_kw={"width_ratios": [...], "height_ratios": [...]}` |
| Nietypowy układ nazwany opisowo | `plt.subplot_mosaic([[...]])` |
| Jedna legenda dla całej figury | `fig.legend(handles=[...], loc=...)` |
| Automatyczne odstępy, bez nachodzenia się elementów | `constrained_layout=True` przy tworzeniu figury |
| Zapis do pliku | `fig.savefig(...)`, nie `plt.savefig(...)` |
| Szybka siatka wprost z `DataFrame` | `df.plot(subplots=True)` |
| Podział wg wartości kolumny kategorycznej | seaborn `FacetGrid` (osobna notatka) |

**Wniosek:** największe realne ryzyko to niespójny kształt `axes` między różnymi układami subplotów (Pułapka 1) — kod, który działał dla siatki 2×2, może się wywalić po zmianie na 1×3. `squeeze=False` i konsekwentne iterowanie przez `axes.flat` (zamiast `axes[i, j]`) eliminują ten problem raz na zawsze.